# Aegis OS - Session 5 Training
## Vehicle Profiles: Drone vs Fixed-Wing Airplane

**Run each block ONE AT A TIME, top to bottom.**

### Before you start:
1. **Runtime > Change Runtime Type > T4 GPU** (must be GPU, not CPU)
2. Upload `session5_colab_package.zip` using the file browser (folder icon on the left sidebar)
3. Upload `latest.pth` to Google Drive at: `My Drive/aegis_checkpoints/latest/latest.pth`


## Block 1 - Unzip Your Package


In [ ]:
from google.colab import files
import zipfile, os

print('Please select session5_colab_package.zip to upload:\n')
uploaded = files.upload()

with zipfile.ZipFile('session5_colab_package.zip', 'r') as zf:
    zf.extractall('.')

for f in ['src/train_rl_session5.py', 'src/cognitive/rl_models.py']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(status + ': ' + f)

print('\nBlock 1 complete!')


## Block 2 - Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')


## Block 3 - Verify Session 4 Brain


In [ ]:
import torch, time, os

path = '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth'

if not os.path.exists(path):
    print('NOT FOUND: ' + path)
    print('Please upload latest.pth to Google Drive at:')
    print('  My Drive / aegis_checkpoints / latest / latest.pth')
else:
    c = torch.load(path, map_location='cpu', weights_only=False)
    print('Checkpoint found!')
    print('  Session:     ' + str(c.get('session')))
    print('  Update:      ' + str(c.get('update')) + ' / 500')
    print('  Total Steps: ' + format(c.get('total_steps', 0), ','))
    print('  Best Reward: ' + str(round(c.get('best_reward', 0), 4)))
    dim = c['actor']['physics_net.0.weight'].shape[1]
    print('  Physics Dim: ' + str(dim) + '  (9 = Session 4 brain, ready for surgery!)')
    saved = time.strftime('%Y-%m-%d %H:%M', time.localtime(c.get('timestamp', 0)))
    print('  Saved:       ' + saved)
    print()
    print('Block 3 complete. Brain verified and ready!')


## Block 4 - Check GPU


In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('GPU Active: ' + name)
    print('VRAM: ' + str(vram) + ' GB')
    print('You are good to start training!')
else:
    print('NO GPU FOUND!')
    print('Go to: Runtime > Change Runtime Type > T4 GPU, then reconnect.')


## Block 5 - START SESSION 5 TRAINING

This runs for up to **5.5 hours** and auto-saves every ~12 minutes to your Drive.

You will see this line confirming the brain upgrade worked:
```
[Surgery] Grafted physics_net.0.weight: expanded (64, 9) -> (64, 10)
```
Then training updates will print every 5 updates like:
```
  Update |       Steps | Avg Reward | Goals/Env
     500 | 524,800,000 |      -0.10 |      0.00
```


In [ ]:
!python src/train_rl_session5.py \
    --session 5 \
    --checkpoint_dir /content/drive/MyDrive/aegis_checkpoints/ \
    --max_hours 5.5


## Block 6 - Save and Check Session 5 Final Brain


In [ ]:
import torch, time, os, shutil

latest_path = '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth'
final_path = '/content/drive/MyDrive/aegis_checkpoints/session5_final.pth'

# Copy the latest brain to a new distinct file for Session 5
if os.path.exists(latest_path):
    shutil.copy(latest_path, final_path)
    print(f'✅ Successfully saved Session 5 brain as: {final_path}\n')
else:
    print('❌ ERROR: Could not find latest.pth to copy!')

c = torch.load(final_path, map_location='cpu', weights_only=False)

print('=== SESSION 5 FINAL RESULTS ===')
print('Session:     ' + str(c.get('session')))
print('Update:      ' + str(c.get('update')) + ' / 3000')
print('Total Steps: ' + format(c.get('total_steps', 0), ','))
print('Best Reward: ' + str(round(c.get('best_reward', 0), 4)))
dim = c['actor']['physics_net.0.weight'].shape[1]
print('Physics Dim: ' + str(dim) + '  (10 = Session 5 brain!)')
saved = time.strftime('%Y-%m-%d %H:%M', time.localtime(c.get('timestamp', 0)))
print('Saved:       ' + saved)
print()
print('NEXT STEP: Download session5_final.pth from Google Drive')
print('           and save it to your computer! (Do NOT overwrite Session 4)')
